In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:

ground_truth[0]

{'question': 'I found this course late — can I still sign up and follow along?',
 'document': '74eb249bbf'}

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [13]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join and follow along. You can start anytime and work through the videos and materials at your own pace.\n\nIf you want a certificate, though, you must submit your project while a live cohort is still accepting submissions.'

In [14]:
assistant.total_cost()

0.00070125

In [15]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [16]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I found this course late — can I still sign up and follow along?',
 'answer_llm': 'Yes — you can still join and follow along. You can start anytime and work through the videos and materials at your own pace.\n\nIf you want a certificate, though, you must submit your project while a live cohort is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [17]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [18]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I found this course late — can I still sign up and follow along?',
 'answer_llm': 'Yes — you can still join and follow along.\n\nIf you want a certificate, though, you need to submit your project while a live cohort is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [19]:
assistant.reset_usage()

In [20]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [21]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/765 [00:00<?, ?it/s]

In [22]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [23]:
assistant.total_cost()

0.8595577500000009

In [27]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [28]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [29]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [30]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [31]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [32]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [33]:
rec = answers[0]

In [34]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [35]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says the student can still join and follow along, and that to receive a certificate they must submit the project while submissions are still being accepted. This preserves the key condition and meaning.', score='good')

In [36]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-6-luna"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [37]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point: the student can still join and follow along, but must submit the project while submissions are open to receive a certificate.', score='good')

In [38]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [39]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/765 [00:00<?, ?it/s]

In [40]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [41]:
df_eval = pd.DataFrame(evaluations)

In [42]:
calc_total_price(usages)

0.6641669999999998

In [43]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 716/765 = 93.59%


In [44]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
4,Can late joiners still submit the project and ...,74eb249bbf,bad,The answer correctly states that submissions m...
13,What’s the best way to ask questions during of...,489dd1c9d9,bad,The answer correctly says to submit live-sessi...
18,Where do I find the main docs and GitHub repo ...,04919992b3,bad,The answer identifies the LLM Zoomcamp docs an...
30,"If I skipped the first homework, can I still e...",9f689c185f,bad,The answer correctly says the skipped homework...
32,What actually matters for getting the certific...,9f689c185f,bad,The AI correctly says the capstone must be com...


In [45]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

In [46]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [47]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [48]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [50]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [51]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [63]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-6-luna")
)

In [64]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [65]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I found this course late — can I still sign up and follow along?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Can students sign up late after the course has started and follow along?"}', call_id='call_J8OifvGTeXBokVtLQVnim8uI', name='search', type='function_call', id='fc_07bbef25ed72114a006abe4325c7dc87d1820400414c188a33', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_J8OifvGTeXBokVtLQVnim8uI',
  'output': '[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs]

In [66]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [67]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"Can students sign up late after the course has started and follow along?"}'}]

In [68]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [71]:

import json


agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": json.dumps(tool_calls),
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I found this course late — can I still sign up and follow along?',
 'answer_agent': 'Yes—you can start whenever you want. The videos and course materials are available online, so you can work through them at your own pace. Check the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for deadlines and submission access; homework can’t be submitted once its form closes. Start with the [course docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and [GitHub repo](https://github.com/DataTalksClub/llm-zoomcamp).',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":\\"Can students sign up late after the course has started and follow along?\\"}"}]',
 'cost': Decimal('0.0002155'),
 'document': '74eb249bbf'}

In [72]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [73]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [74]:
df_agent = pd.DataFrame(agent_answers)

In [75]:
df_agent["cost"].sum()

Decimal('0.0084180')

In [77]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [78]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [79]:
agent_answers[0]

{'question': 'I found this course late — can I still sign up and follow along?',
 'answer_agent': 'Yes—you can join late and start whenever you like. The videos and course materials are available online; start with the [course docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and [GitHub repo](https://github.com/DataTalksClub/llm-zoomcamp), and check the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for deadlines.\n\nIf you want a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is still accepting them.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":\\"late enrollment sign up after course started follow along late join\\"}"}]',
 'cost': 0.0001962,
 'document': '74eb249bbf'}

In [80]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [81]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [82]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [83]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth on the core point: you can join late and follow along, but to get a certificate you must submit the project while submissions are still open. The extra details about course docs/GitHub/course platform are not in the original answer but do not conflict with it. The mention of peer reviews and a live cohort is slightly more specific than the ground truth, but still aligned with the certificate requirement.', answer_score='good', trajectory_reasoning='The search query was relevant to the question and included the key idea of late enrollment/following along. Only one search call was made, which is reasonable for this simple FAQ-style question. The tool call supported the final answer.', trajectory_score='good')

In [84]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [85]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [86]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [87]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [88]:
calc_total_price(usages)

0.05379750000000001

In [89]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    48
bad      2
Name: count, dtype: int64

In [90]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [91]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)